# MSI 5001 Introduction to AI: Concepts, Applications, and Evaluation (Capstone Project)

## Preprocessing Step

- **Group:** 26
- **Members:** Alkaff, Dev, Kee Fong


## Overview

- **Pipeline 1:** Generate MFCC - `process_mel_split()`
- **Pipeline 2:** Generate Spectrographs - `process_mfcc_split()`
- **Pipeline 3:** Downsample to 24 kHz - `process_wav24_split()`


## Import Statements and Setup


In [1]:
# Import Statements
# -----------------

import os
from pathlib import Path
import shutil
import re

from tqdm import tqdm
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from scipy.signal import resample_poly
import soundfile as sf
import librosa


In [2]:
# Global Config
# -------------

BASE_DIR = Path("D:\\MSI5001 Datasets\\Urban Sound")
SPLITS = ["Train", "Test"]

RAW_ROOT = Path("raw")
OUT_MEL_ROOT = Path("mel_spectro")
OUT_WAV24_ROOT = Path("wave_24k")
OUT_MFCC_ROOT = Path("mfcc")


In [3]:
# Process Config
# --------------

# Loudness normalization
TARGET_RMS_DBFS = -20.0                 # target loudness (~broadcast-light). Change if you prefer
TARGET_RMS = 10 ** (TARGET_RMS_DBFS / 20)
PEAK_LIMIT = 0.999                      # anti-clipping headroom
EPS = 1e-12

# Mel-spectrogram params (for 24 kHz or original sr; we use normalized audio at its native sr)
N_MELS = 128
N_FFT = 1024
HOP = 512
FMIN = 20.0
# We'll cap fmax to 0.45 * sr at runtime for safety (Nyquist-guard); typical 10–11 kHz at 24k

# MFCC params
N_MFCC = 40  # includes c0

# Downsample target
TARGET_SR = 24000

SECONDS = 4.0  # keep this consistent with your 4s pipeline
TARGET_SAMPLES = int(SECONDS * TARGET_SR)  # 96000
# librosa's frame count: 1 + floor((N - n_fft) / hop_length)
TARGET_FRAMES = 1 + (TARGET_SAMPLES - N_FFT) // HOP  # typically 186 for 4s, 24k, 1024/512


In [5]:
# Helper Functions
# ----------------

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def iter_wav_files(in_dir: Path):
    for root, _, files in os.walk(in_dir):
        for f in files:
            if f.lower().endswith(".wav"):
                yield Path(root) / f

def to_mono(y):
    return y if y.ndim == 1 else np.mean(y, axis=1)

def rms(y):
    return float(np.sqrt(np.mean(y ** 2) + EPS))

def pad_or_trim_to_seconds(y, sr, seconds=4.0, mode="repeat"):
    target = int(seconds * sr)
    L = len(y)
    if L == target:
        return y.astype(np.float32)
    if L > target:
        # center crop (deterministic in preprocessing)
        start = (L - target) // 2
        return y[start:start+target].astype(np.float32)
    # pad
    pad_len = target - L
    if mode == "repeat" and L > 0:
        reps = int(np.ceil(target / L))
        return np.tile(y, reps)[:target].astype(np.float32)
    elif mode == "reflect" and L > 0:
        return np.pad(y, (0, pad_len), mode="reflect").astype(np.float32)
    else:
        return np.pad(y, (0, pad_len), mode="constant").astype(np.float32)

def loudness_normalize(y):
    cur_rms = rms(y)
    if cur_rms < EPS:
        return y, 1.0
    gain = TARGET_RMS / cur_rms
    peak = np.max(np.abs(y))
    if peak * gain > PEAK_LIMIT:
        gain = PEAK_LIMIT / (peak + EPS)
    return (y * gain).astype(np.float32), gain

def downsample(y, sr_in):
    if sr_in == TARGET_SR:
        return y.astype(np.float32)
    y_ds = resample_poly(y, up=TARGET_SR, down=sr_in)
    peak = np.max(np.abs(y_ds)) + EPS
    if peak > 1.0:
        y_ds /= peak
    return y_ds.astype(np.float32)

def compute_logmel(y, sr):
    fmax = min(0.45 * sr, sr/2 - 1.0)
    S = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=N_FFT, hop_length=HOP,
        n_mels=N_MELS, fmin=FMIN, fmax=fmax, power=2.0
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    return S_db.astype(np.float32)

def compute_mfcc_stack(y, sr):
    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP
    )  # (n_mfcc, T)
    T = mfcc.shape[1]

    # Pick an odd width <= T (default target 9)
    target = 9
    width = target if T >= target else (T if (T % 2 == 1) else T - 1)

    if width >= 3:
        # Use mode='nearest' to be robust at short lengths
        d1 = librosa.feature.delta(mfcc, order=1, width=width, mode='nearest')
        d2 = librosa.feature.delta(mfcc, order=2, width=width, mode='nearest')
    else:
        # Too few frames to compute deltas reliably -> fallback
        d1 = np.zeros_like(mfcc, dtype=np.float32)
        d2 = np.zeros_like(mfcc, dtype=np.float32)

    return np.vstack([mfcc.astype(np.float32), d1.astype(np.float32), d2.astype(np.float32)])

def save_png(path: Path, mel_db: np.ndarray, resize_to: Optional[tuple] = None):
    """
    Save a mel spectrogram as a grayscale PNG with a fixed time width (TARGET_FRAMES).
    Optionally resize to (H, W) AFTER fixing the width.

    - mel_db: shape (n_mels, T)
    - resize_to: optional (H, W). Use (224, 224) if you want square for pretrained CNNs.
    """
    ensure_dir(path.parent)

    # 1) Force fixed time width by crop/pad along TIME only (axis=1)
    n_mels, T = mel_db.shape
    # Crop (center) if longer
    if T > TARGET_FRAMES:
        start = (T - TARGET_FRAMES) // 2
        mel_db = mel_db[:, start:start + TARGET_FRAMES]
    # Pad (right) if shorter — pad with low-energy value to mimic silence
    elif T < TARGET_FRAMES:
        pad_w = TARGET_FRAMES - T
        pad_val = float(mel_db.min())  # silence region in dB
        mel_db = np.pad(mel_db, ((0, 0), (0, pad_w)), mode='constant', constant_values=pad_val)

    # 2) Normalize to [0, 255] uint8
    mel_min, mel_max = mel_db.min(), mel_db.max()
    mel_norm = 255.0 * (mel_db - mel_min) / (mel_max - mel_min + EPS)
    mel_img = mel_norm.astype(np.uint8)  # (n_mels, TARGET_FRAMES)

    # 3) Optional resize for pretrained backbones
    #    Note: PIL expects (H, W) as (rows, cols) which matches (n_mels, frames)
    img = Image.fromarray(mel_img, mode='L')  # grayscale
    if resize_to is not None:
        img = img.resize((resize_to[1], resize_to[0]), resample=Image.BILINEAR)  # (W, H)

    # 4) Save
    img.save(path)

def parse_class_from_filename(name: str) -> Optional[str]:
    # Extract class after the last '__' (e.g., 'Test_4__car_horn.wav' -> 'car_horn').
    idx = name.rfind("__")
    if idx == -1:
        return None
    base = name[idx + 2:]                # 'car_horn.wav'
    base = re.sub(r"\.[^.]+$", "", base) # remove extension -> 'car_horn'
    return base.strip() or None

def save_npy(path: Path, arr: np.ndarray):
    ensure_dir(path.parent)
    np.save(path, arr)

def save_wav(path: Path, y: np.ndarray, sr: int):
    ensure_dir(path.parent)
    sf.write(path, y, sr)


## Organise the Dataset Folder Structure

- Currently, all the files are dumped into the directories `Test\*` and `Train\*` (Flat file structure)
- We will organises the files so that it will be `Test\classname\*` and `Train\classname\*`

**NOTE:** Run only if the Train and Test folders are flat


In [7]:
# Organise Files
# --------------

"""
Move files in raw/<split>/ into per-class subfolders based on filename.
Expects filename: Something__CLASS.wav
"""
for split in SPLITS:
    split_dir = BASE_DIR / RAW_ROOT / split
    if not split_dir.exists():
        print(f"[SKIP] Missing: {split_dir}")
        continue

    files = [p for p in split_dir.glob("*") if p.is_file() and p.suffix.lower() == ".wav"]
    moved, skipped = 0, 0

    for f in files:
        cls = parse_class_from_filename(f.name)
        if not cls:
            print(f"[SKIP] Cannot parse class from: {f.name}")
            skipped += 1
            continue

        dst_dir = split_dir / cls
        dst_dir.mkdir(parents=True, exist_ok=True)

        dst_path = dst_dir / f.name
        if dst_path.exists():
            # Avoid overwriting accidental duplicates
            i, stem, suf = 1, f.stem, f.suffix
            while dst_path.exists():
                dst_path = dst_dir / f"{stem}__dup{i}{suf}"
                i += 1

        shutil.move(str(f), str(dst_path))
        moved += 1

    print(f"[raw/{split}] moved={moved}, skipped={skipped}")


[raw/Train] moved=1760, skipped=0
[raw/Test] moved=837, skipped=0


## Assemble into Pipelines

### Pipeline 1: Generate MFCC - `process_mel_split()`

1. **Load** Train/Test from `raw`
2. **Normalize audio** so that everything is similar loudness
3. **Calculate** the **MFCC** (40 features) and their 1st and 2nd derivatives (total 120 features)
    - 1st derivative and 2nd derivative gives information about the slope of the curve at the timestep
    - Additional information that will be useful as features for the subsequent steps
4. **Save** the generated data as tensors (`.npy`) for train/test (`mfcc`)

### Pipeline 2: Generate Spectrographs - `process_mfcc_split()`

1. **Load** Train/Test from `raw`
2. **Normalize audio** so that everything is similar loudness
3. Do **repeat-fill** or **center-crop** to ensure all clips are `4s` long
4. **Calculate** mel-spectrograph for 128 bins
    - Use `librosa.feature.melspectrogram()` to calculate the 128 mel-bins of the frequency range
    - Convert it to log-scale using `librosa.power_to_db()` so that the higher frequencies will be scaled down more 
5. Convert to spectrograph images of 186x128 pixels
    - Height 128 → 128 bins f
    - Width 186 → 186 timesteps from 4s audio clip
6. **Save** the generated data as `.png` files for train/test (`mel_spectro`)

### Pipeline 3: Downsample to 24 kHz - `process_wav24_split()`

1. **Load** Train/Test from `raw`
2. **Normalize audio** so that everything is similar loudness
3. Do **repeat-fill** or **center-crop** to ensure all clips are `4s` long
4. **Convert** original wave **to 24 kHz** for train/test (`wave_24k`)


In [8]:
# 3 Pipelines
# -----------

# 1) MEL (PNG) per split

def process_mel_split(split: str):
    in_dir = BASE_DIR / RAW_ROOT / split
    out_dir = BASE_DIR / OUT_MEL_ROOT / split
    wav_files = list(iter_wav_files(in_dir))
    print(f"[MEL:{split}] Found {len(wav_files)} files")

    for wav_path in tqdm(wav_files, desc=f"MEL {split}"):
        y, sr = sf.read(wav_path, always_2d=False)
        y = to_mono(y).astype(np.float32)
        y_norm, _ = loudness_normalize(y)

        # resample → 24k, then pad/trim to 4.0s (ensures identical mel width)
        y_24k = y_norm
        y_24k_4s = pad_or_trim_to_seconds(y_24k, TARGET_SR, seconds=4.0, mode="repeat")

        # mel from 4s @ 24k
        mel_db = compute_logmel(y_24k_4s, TARGET_SR)

        rel = wav_path.relative_to(in_dir)
        stem = rel.stem
        out_png = out_dir / rel.parent / f"{stem}.png"
        save_png(out_png, mel_db)

    print(f"[MEL:{split}] Done. PNGs saved to {out_dir}")


# 2) WAV 24 kHz per split

def process_wav24_split(split: str):
    in_dir = BASE_DIR / RAW_ROOT / split
    out_dir = BASE_DIR / OUT_WAV24_ROOT / split
    wav_files = list(iter_wav_files(in_dir))
    print(f"[WAV24:{split}] Found {len(wav_files)} files")

    for wav_path in tqdm(wav_files, desc=f"WAV24 {split}"):
        y, sr = sf.read(wav_path, always_2d=False)
        y = to_mono(y).astype(np.float32)
        y_norm, _ = loudness_normalize(y)

        # resample → 24k, then pad/trim to 4.0s
        y_24k = downsample(y_norm, sr_in=sr)
        y_24k_4s = pad_or_trim_to_seconds(y_24k, TARGET_SR, seconds=4.0, mode="repeat")

        rel = wav_path.relative_to(in_dir)
        stem = rel.stem
        out_wav = out_dir / rel.parent / f"{stem}.wav"
        save_wav(out_wav, y_24k_4s, TARGET_SR)

    print(f"[WAV24:{split}] Done. WAVs saved to {out_dir}")


# 3) MFCC (+Δ +ΔΔ) per split

def process_mfcc_split(split: str):
    in_dir = BASE_DIR / RAW_ROOT / split
    out_dir = BASE_DIR / OUT_MFCC_ROOT / split  # saving MFCC npy beside mel PNGs
    wav_files = list(iter_wav_files(in_dir))
    print(f"[MFCC:{split}] Found {len(wav_files)} files")

    for wav_path in tqdm(wav_files, desc=f"MFCC {split}"):
        y, sr = sf.read(wav_path, always_2d=False)
        y = to_mono(y).astype(np.float32)
        y_norm, _ = loudness_normalize(y)

        mfcc_stack = compute_mfcc_stack(y_norm, sr)

        rel = wav_path.relative_to(in_dir)
        stem = rel.stem
        out_npy = out_dir / rel.parent / f"{stem}.npy"
        save_npy(out_npy, mfcc_stack)

    print(f"[MFCC:{split}] Done. NPYs saved to {out_dir}")


## RUN THE PIPELINES!

In [13]:
# !!! RUN !!!
# -----------

for split in SPLITS:
    process_mel_split(split)

for split in SPLITS:
    process_wav24_split(split)

for split in SPLITS:
    process_mfcc_split(split)

print("✅ All processing complete.")

[MEL:Train] Found 1760 files


MEL Train: 100%|██████████████████████████████████████████████████████████████████| 1760/1760 [00:15<00:00, 110.97it/s]


[MEL:Train] Done. PNGs saved to D:\MSI5001 Datasets\Urban Sound\mel_spectro\Train
[MEL:Test] Found 837 files


MEL Test: 100%|█████████████████████████████████████████████████████████████████████| 837/837 [00:07<00:00, 110.50it/s]


[MEL:Test] Done. PNGs saved to D:\MSI5001 Datasets\Urban Sound\mel_spectro\Test
[WAV24:Train] Found 1760 files


...

[WAV24:Test] Done. WAVs saved to D:\MSI5001 Datasets\Urban Sound\wave_24k\Test
[MFCC:Train] Found 1760 files


MFCC Train: 100%|█████████████████████████████████████████████████████████████████| 1760/1760 [00:17<00:00, 102.51it/s]


[MFCC:Train] Done. NPYs saved to D:\MSI5001 Datasets\Urban Sound\mfcc\Train
[MFCC:Test] Found 837 files


MFCC Test: 100%|████████████████████████████████████████████████████████████████████| 837/837 [00:08<00:00, 101.35it/s]

[MFCC:Test] Done. NPYs saved to D:\MSI5001 Datasets\Urban Sound\mfcc\Test
✅ All processing complete.
